<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Heisenberg's lab
</font>
</h1>

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Dataset
</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

</font>
</p>


<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
In this section, <code>import</code> your required libraries and tools, and read the data files saved in the <code>Data</code> folder under the names <code>train.csv</code> and <code>test.csv</code>, and load them into your workspace.
</font>
</p>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'\nTarget columns: Marijuana, LSD, Mushroom, Psychotropic, Ex')
print(f'\nTarget distributions:')
for col in ['Marijuana', 'LSD', 'Mushroom', 'Psychotropic', 'Ex']:
    print(f'{col}: {train_df[col].value_counts().sort_index().to_dict()}')

Train shape: (1319, 18)
Test shape: (566, 13)

Target columns: Marijuana, LSD, Mushroom, Psychotropic, Ex

Target distributions:
Marijuana: {0: 620, 1: 250, 2: 449}
LSD: {0: 1055, 1: 212, 2: 52}
Mushroom: {0: 1020, 1: 268, 2: 31}
Psychotropic: {0: 924, 1: 300, 2: 95}
Ex: {0: 957, 1: 299, 2: 63}


<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Preprocessing and Feature Engineering
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;In this question, you can use any preprocessing/feature engineering technique of your choice.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;The techniques you use will <b>not</b> be directly evaluated by the judging system. Rather, they will all impact the accuracy of your model; therefore, the better preprocessing/feature engineering you perform to improve the model's accuracy, the more points you will earn from this question.
&nbsp;&nbsp;&nbsp;&nbsp;In this section, you can allocate a portion of the available data for validation.
</font>
</p>

In [2]:
feature_cols = ['Age', 'Sex', 'EducationLevel', 'Country', 'Background',
                'EmotionalStabilityScore', 'SocialEnergyScore', 'OpennessScore',
                'CooperationScore', 'SelfDisciplineScore', 'ImpulseControlScore',
                'NoveltySeekingScore']
target_cols = ['Marijuana', 'LSD', 'Mushroom', 'Psychotropic', 'Ex']

X = train_df[feature_cols].copy()
y = train_df[target_cols].copy()
X_test = test_df[feature_cols].copy()

# Feature interactions
X['Emotional_x_Impulse'] = X['EmotionalStabilityScore'] * X['ImpulseControlScore']
X['Social_x_Novelty'] = X['SocialEnergyScore'] * X['NoveltySeekingScore']
X['Openness_x_Novelty'] = X['OpennessScore'] * X['NoveltySeekingScore']
X['Cooperation_x_SelfDiscipline'] = X['CooperationScore'] * X['SelfDisciplineScore']
X['Age_x_SelfDiscipline'] = X['Age'] * X['SelfDisciplineScore']
X['Emotional_x_Social'] = X['EmotionalStabilityScore'] * X['SocialEnergyScore']
X['Impulse_x_Novelty'] = X['ImpulseControlScore'] * X['NoveltySeekingScore']
X['Openness_x_Cooperation'] = X['OpennessScore'] * X['CooperationScore']

X_test['Emotional_x_Impulse'] = X_test['EmotionalStabilityScore'] * X_test['ImpulseControlScore']
X_test['Social_x_Novelty'] = X_test['SocialEnergyScore'] * X_test['NoveltySeekingScore']
X_test['Openness_x_Novelty'] = X_test['OpennessScore'] * X_test['NoveltySeekingScore']
X_test['Cooperation_x_SelfDiscipline'] = X_test['CooperationScore'] * X_test['SelfDisciplineScore']
X_test['Age_x_SelfDiscipline'] = X_test['Age'] * X_test['SelfDisciplineScore']
X_test['Emotional_x_Social'] = X_test['EmotionalStabilityScore'] * X_test['SocialEnergyScore']
X_test['Impulse_x_Novelty'] = X_test['ImpulseControlScore'] * X_test['NoveltySeekingScore']
X_test['Openness_x_Cooperation'] = X_test['OpennessScore'] * X_test['CooperationScore']

# Quadratic features
X['Emotional_sq'] = X['EmotionalStabilityScore'] ** 2
X['Impulse_sq'] = X['ImpulseControlScore'] ** 2
X['Novelty_sq'] = X['NoveltySeekingScore'] ** 2
X_test['Emotional_sq'] = X_test['EmotionalStabilityScore'] ** 2
X_test['Impulse_sq'] = X_test['ImpulseControlScore'] ** 2
X_test['Novelty_sq'] = X_test['NoveltySeekingScore'] ** 2

print(f'Features shape: {X.shape}')

Features shape: (1319, 23)


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Model Training
</font>
</h2>



In [3]:
# TODO:
models = {}

for target in target_cols:
    print(f'\nTraining models for {target}...')
    y_target = y[target].values
    
    # XGBoost
    xgb_params = {
        'objective': 'multi:softmax',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_estimators': 500,
        'verbosity': 0
    }
    xgb_model = xgb.XGBClassifier(**xgb_params)
    
    # LightGBM
    lgb_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'max_depth': 7,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_estimators': 500,
        'verbosity': -1
    }
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    
    # CatBoost
    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        random_seed=42,
        verbose=0,
        loss_function='MultiClass'
    )
    
    # Cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    xgb_scores = []
    lgb_scores = []
    cat_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_target)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_target[train_idx], y_target[val_idx]
        
        xgb_model_clone = xgb.XGBClassifier(**xgb_params)
        lgb_model_clone = lgb.LGBMClassifier(**lgb_params)
        cat_model_clone = CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, random_seed=42, verbose=0,
            loss_function='MultiClass'
        )
        
        xgb_model_clone.fit(X_train, y_train)
        lgb_model_clone.fit(X_train, y_train)
        cat_model_clone.fit(X_train, y_train)
        
        xgb_pred = xgb_model_clone.predict(X_val)
        lgb_pred = lgb_model_clone.predict(X_val)
        cat_pred = cat_model_clone.predict(X_val).flatten()
        
        xgb_scores.append(f1_score(y_val, xgb_pred, average='macro'))
        lgb_scores.append(f1_score(y_val, lgb_pred, average='macro'))
        cat_scores.append(f1_score(y_val, cat_pred, average='macro'))
    
    print(f'  XGBoost CV F1: {np.mean(xgb_scores):.4f} (+/- {np.std(xgb_scores):.4f})')
    print(f'  LightGBM CV F1: {np.mean(lgb_scores):.4f} (+/- {np.std(lgb_scores):.4f})')
    print(f'  CatBoost CV F1: {np.mean(cat_scores):.4f} (+/- {np.std(cat_scores):.4f})')
    
    # Select best model
    scores = {'xgb': np.mean(xgb_scores), 'lgb': np.mean(lgb_scores), 'cat': np.mean(cat_scores)}
    best = max(scores, key=scores.get)
    print(f'  Best model: {best}')
    
    # Train best on full data
    if best == 'xgb':
        final_model = xgb.XGBClassifier(**xgb_params)
    elif best == 'lgb':
        final_model = lgb.LGBMClassifier(**lgb_params)
    else:
        final_model = CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, random_seed=42, verbose=0,
            loss_function='MultiClass'
        )
    
    final_model.fit(X, y_target)
    models[target] = final_model

print('\nAll models trained!')


Training models for Marijuana...


  XGBoost CV F1: 0.9434 (+/- 0.0120)
  LightGBM CV F1: 0.9410 (+/- 0.0124)
  CatBoost CV F1: 0.9446 (+/- 0.0121)
  Best model: cat



Training models for LSD...


  XGBoost CV F1: 0.5131 (+/- 0.0377)
  LightGBM CV F1: 0.5027 (+/- 0.0299)
  CatBoost CV F1: 0.4837 (+/- 0.0178)
  Best model: xgb



Training models for Mushroom...


  XGBoost CV F1: 0.4917 (+/- 0.0511)
  LightGBM CV F1: 0.4840 (+/- 0.0499)
  CatBoost CV F1: 0.4776 (+/- 0.0309)
  Best model: xgb



Training models for Psychotropic...


  XGBoost CV F1: 0.5011 (+/- 0.0525)
  LightGBM CV F1: 0.5084 (+/- 0.0529)
  CatBoost CV F1: 0.4895 (+/- 0.0124)
  Best model: lgb



Training models for Ex...


  XGBoost CV F1: 0.9821 (+/- 0.0147)
  LightGBM CV F1: 0.9828 (+/- 0.0153)
  CatBoost CV F1: 0.9807 (+/- 0.0162)
  Best model: lgb

All models trained!


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Evaluation Metric
</font>
</h2>



In [4]:
# TODO:
print('=== Evaluation on Training Data ===')
total_score = 0
for target in target_cols:
    model = models[target]
    y_pred = model.predict(X)
    macro_f1 = f1_score(y[target], y_pred, average='macro')
    print(f'{target}: Macro F1 = {macro_f1:.4f}')
    
    if macro_f1 >= 0.66:
        scaled = min(1, max(0, (macro_f1 - 0.50) / 0.50))
        points = 20 * scaled
        print(f'  Points earned: {points:.2f}/20')
    else:
        print(f'  Points earned: 0/20 (below 0.66 threshold)')

avg_f1 = np.mean([f1_score(y[t], models[t].predict(X), average='macro') for t in target_cols])
print(f'\nAverage Macro F1: {avg_f1:.4f}')

=== Evaluation on Training Data ===


Marijuana: Macro F1 = 0.9818
  Points earned: 19.27/20
LSD: Macro F1 = 1.0000
  Points earned: 20.00/20
Mushroom: Macro F1 = 1.0000
  Points earned: 20.00/20
Psychotropic: Macro F1 = 1.0000
  Points earned: 20.00/20
Ex: Macro F1 = 1.0000
  Points earned: 20.00/20



Average Macro F1: 0.9964


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Prediction on Test Data
</font>
</h2>


In [5]:
# TODO:
predictions = {}
for target in target_cols:
    predictions[target] = models[target].predict(X_test)

submission = pd.DataFrame()
submission['ID'] = test_df['ID']
for target in target_cols:
    submission[target] = predictions[target].astype(int)

print('Submission shape:', submission.shape)
print(submission.head(10))
print('\nPrediction distributions:')
for target in target_cols:
    print(f'{target}: {submission[target].value_counts().sort_index().to_dict()}')

Submission shape: (566, 6)
   ID  Marijuana  LSD  Mushroom  Psychotropic  Ex
0   1          0    0         0             0   0
1   2          0    0         0             0   0
2   3          0    0         0             0   2
3   4          0    0         0             0   0
4   5          0    0         0             0   0
5   6          0    0         0             0   0
6   7          0    0         0             0   0
7   8          0    0         0             0   0
8   9          0    0         0             0   0
9  10          0    0         0             0   0

Prediction distributions:
Marijuana: {0: 472, 1: 26, 2: 68}
LSD: {0: 525, 1: 41}
Mushroom: {0: 523, 1: 42, 2: 1}
Psychotropic: {0: 468, 1: 87, 2: 11}
Ex: {0: 416, 1: 125, 2: 25}


<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;Run the cell below to generate the <code>result.zip</code> file. Please note that you must save the changes made in the notebook (<code>ctrl+s</code>) before running the cell below; otherwise, your score will be changed to zero at the end of the competition.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;Also, if you are using Colab to run this notebook file, download the latest version of your notebook and place it inside the submission file before submitting the <code>result.zip</code> file.
</font>
</p>

In [6]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'Drug.ipynb')):
    %notebook -e Drug.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['Drug.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['Drug.ipynb', 'submission.csv']
